# 📈 Sales Prediction
### CodeAlpha Machine Learning Internship — Task 4
**Intern:** Raj Chakrawarti | **ID:** CA/DF1/68634

---
## 📌 Objective
Predict product sales based on advertising spend across:
- TV, Radio, Newspaper advertising budgets
- Build regression models to forecast sales
- Find which advertising channel gives best ROI

## 📦 Step 1: Import Libraries

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

plt.rcParams['figure.figsize'] = (12, 6)
sns.set_style('whitegrid')
print('✅ Libraries imported!')

## 📂 Step 2: Load & Explore Dataset

In [ ]:
np.random.seed(42)
n = 500

TV        = np.random.uniform(0.7, 296, n)
Radio     = np.random.uniform(0.0, 49.6, n)
Newspaper = np.random.uniform(0.3, 114, n)

# Sales depends mostly on TV & Radio
Sales = (
    2.9
    + 0.046 * TV
    + 0.188 * Radio
    + 0.001 * Newspaper
    + 0.0011 * TV * Radio
    + np.random.normal(0, 1.5, n)
)
Sales = np.clip(Sales, 1.6, 30)

df = pd.DataFrame({
    'TV': np.round(TV, 1),
    'Radio': np.round(Radio, 1),
    'Newspaper': np.round(Newspaper, 1),
    'Sales': np.round(Sales, 1)
})

print('📊 Dataset Shape:', df.shape)
print('\n🔍 First 10 rows:')
df.head(10)

In [ ]:
print('📋 Info:')
df.info()
print('\n📈 Summary:')
df.describe().round(2)

In [ ]:
print('🔎 Missing Values:')
print(df.isnull().sum())

## 📊 Step 3: EDA

In [ ]:
# Distribution of all columns
fig, axes = plt.subplots(2, 2, figsize=(13, 9))
cols   = ['TV', 'Radio', 'Newspaper', 'Sales']
colors = ['#3498db', '#e74c3c', '#2ecc71', '#f39c12']
for ax, col, color in zip(axes.flatten(), cols, colors):
    ax.hist(df[col], bins=30, color=color, edgecolor='white', alpha=0.85)
    ax.set_title(f'{col} Distribution', fontweight='bold')
    ax.set_xlabel(col)
    ax.set_ylabel('Frequency')
plt.suptitle('📊 Feature Distributions', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Scatter: Each Feature vs Sales
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
channels = ['TV', 'Radio', 'Newspaper']
colors   = ['#3498db', '#e74c3c', '#2ecc71']

for ax, ch, color in zip(axes, channels, colors):
    ax.scatter(df[ch], df['Sales'], alpha=0.4, color=color, s=20)
    # Trendline
    z = np.polyfit(df[ch], df['Sales'], 1)
    p = np.poly1d(z)
    x_line = np.linspace(df[ch].min(), df[ch].max(), 100)
    ax.plot(x_line, p(x_line), 'k--', linewidth=1.5)
    corr = df[ch].corr(df['Sales'])
    ax.set_title(f'{ch} vs Sales\nCorr: {corr:.3f}', fontweight='bold')
    ax.set_xlabel(f'{ch} Budget ($000s)')
    ax.set_ylabel('Sales (000 units)')

plt.suptitle('📺 Advertising Budget vs Sales', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Correlation Heatmap
plt.figure(figsize=(7, 5))
sns.heatmap(df.corr(), annot=True, fmt='.3f', cmap='coolwarm',
            square=True, linewidths=0.5)
plt.title('🔥 Correlation Heatmap', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Budget Share Analysis
total_budget = df[['TV','Radio','Newspaper']].sum()
plt.figure(figsize=(7, 7))
plt.pie(total_budget, labels=total_budget.index,
        autopct='%1.1f%%', colors=['#3498db','#e74c3c','#2ecc71'],
        startangle=90, explode=(0.05,0.05,0.05), shadow=True)
plt.title('📺 Total Advertising Budget Share', fontsize=14, fontweight='bold')
plt.show()

In [ ]:
# Sales vs TV — colored by Radio budget
plt.figure(figsize=(10, 6))
sc = plt.scatter(df['TV'], df['Sales'], c=df['Radio'],
                 cmap='YlOrRd', alpha=0.6, s=30)
plt.colorbar(sc, label='Radio Budget ($000s)')
plt.title('📺 TV Budget vs Sales (colored by Radio)', fontsize=14, fontweight='bold')
plt.xlabel('TV Budget ($000s)')
plt.ylabel('Sales (000 units)')
plt.tight_layout()
plt.show()

## ⚙️ Step 4: Preprocessing

In [ ]:
X = df[['TV', 'Radio', 'Newspaper']]
y = df['Sales']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)

print(f'✅ Train: {X_train.shape[0]} | Test: {X_test.shape[0]} | Features: 3')

## 🤖 Step 5: Model Training & Comparison

In [ ]:
models = {
    'Linear Regression' : LinearRegression(),
    'Ridge'             : Ridge(alpha=1.0),
    'Lasso'             : Lasso(alpha=0.01),
    'Decision Tree'     : DecisionTreeRegressor(random_state=42),
    'Random Forest'     : RandomForestRegressor(n_estimators=100, random_state=42),
    'Gradient Boosting' : GradientBoostingRegressor(random_state=42)
}

results = {}
for name, model in models.items():
    model.fit(X_train_sc, y_train)
    y_pred = model.predict(X_test_sc)
    r2   = r2_score(y_test, y_pred)
    mae  = mean_absolute_error(y_test, y_pred)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    results[name] = {'R² Score': round(r2,4), 'MAE': round(mae,4), 'RMSE': round(rmse,4)}
    print(f'✅ {name:22s} | R²: {r2:.4f} | MAE: {mae:.4f} | RMSE: {rmse:.4f}')

results_df = pd.DataFrame(results).T.sort_values('R² Score', ascending=False)
print('\n🏆 Model Comparison:')
results_df

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

bars = axes[0].bar(results_df.index, results_df['R² Score'],
                   color=['#e74c3c']+['#3498db']*(len(results_df)-1), edgecolor='white')
for bar, val in zip(bars, results_df['R² Score']):
    axes[0].text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.005,
                 f'{val:.4f}', ha='center', fontsize=9, fontweight='bold')
axes[0].set_title('R² Score Comparison', fontweight='bold')
axes[0].set_ylabel('R² Score')
axes[0].tick_params(axis='x', rotation=20)
axes[0].set_ylim(0, 1.1)

axes[1].bar(results_df.index, results_df['RMSE'], color='#9b59b6', edgecolor='white')
axes[1].set_title('RMSE Comparison (lower = better)', fontweight='bold')
axes[1].set_ylabel('RMSE')
axes[1].tick_params(axis='x', rotation=20)

plt.suptitle('🤖 Model Comparison', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 📈 Step 6: Best Model Evaluation

In [ ]:
best_model   = models['Gradient Boosting']
y_pred_best  = best_model.predict(X_test_sc)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].scatter(y_test, y_pred_best, alpha=0.5, color='#3498db', s=25)
axes[0].plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', lw=2)
axes[0].set_title('Actual vs Predicted Sales', fontweight='bold')
axes[0].set_xlabel('Actual Sales')
axes[0].set_ylabel('Predicted Sales')

residuals = y_test - y_pred_best
axes[1].hist(residuals, bins=35, color='#e74c3c', edgecolor='white')
axes[1].axvline(0, color='black', linestyle='--')
axes[1].set_title('Residual Distribution', fontweight='bold')
axes[1].set_xlabel('Residual')

plt.suptitle('📈 Gradient Boosting — Evaluation', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print(f'🎯 R² Score : {r2_score(y_test, y_pred_best):.4f}')
print(f'📉 MAE      : {mean_absolute_error(y_test, y_pred_best):.4f}')
print(f'📉 RMSE     : {np.sqrt(mean_squared_error(y_test, y_pred_best)):.4f}')

In [ ]:
# Feature Importance
feat_imp = pd.Series(best_model.feature_importances_, index=['TV','Radio','Newspaper'])
plt.figure(figsize=(8, 4))
colors = ['#3498db','#e74c3c','#2ecc71']
bars = plt.bar(feat_imp.index, feat_imp.values, color=colors, edgecolor='white', width=0.4)
for bar, val in zip(bars, feat_imp.values):
    plt.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.005,
             f'{val:.4f}', ha='center', fontweight='bold')
plt.title('🔍 Feature Importance — Which Channel Drives Sales?', fontsize=13, fontweight='bold')
plt.ylabel('Importance Score')
plt.tight_layout()
plt.show()

## 🔮 Step 7: Sales Forecast

In [ ]:
# ROI Analysis: What if we increase TV budget?
tv_budgets = np.linspace(0, 300, 100)
radio_fix  = 25
news_fix   = 30

test_data = pd.DataFrame({
    'TV': tv_budgets,
    'Radio': [radio_fix]*100,
    'Newspaper': [news_fix]*100
})
test_scaled   = scaler.transform(test_data)
sales_forecast= best_model.predict(test_scaled)

plt.figure(figsize=(10, 5))
plt.plot(tv_budgets, sales_forecast, color='#e74c3c', linewidth=2.5)
plt.fill_between(tv_budgets, sales_forecast, alpha=0.1, color='#e74c3c')
plt.title('📺 TV Budget vs Predicted Sales Forecast', fontsize=14, fontweight='bold')
plt.xlabel('TV Budget ($000s)')
plt.ylabel('Predicted Sales (000 units)')
plt.tight_layout()
plt.show()

# Predict for a new campaign
new_campaign = pd.DataFrame([{'TV': 150, 'Radio': 30, 'Newspaper': 20}])
new_sc       = scaler.transform(new_campaign)
pred_sales   = best_model.predict(new_sc)[0]
print(f'\n🔮 Campaign Prediction:')
print(f'   TV: $150K | Radio: $30K | Newspaper: $20K')
print(f'   Predicted Sales: {pred_sales:.2f} (000 units)')

## 📝 Step 8: Summary

In [ ]:
best_r2   = results_df['R² Score'].iloc[0]
best_name = results_df.index[0]

tv_corr   = df['TV'].corr(df['Sales'])
rad_corr  = df['Radio'].corr(df['Sales'])
news_corr = df['Newspaper'].corr(df['Sales'])

print('=' * 60)
print('📈 SALES PREDICTION — FINAL SUMMARY')
print('=' * 60)
print(f'📁 Dataset     : Advertising Sales Dataset (500 samples)')
print(f'🎯 Features    : TV, Radio, Newspaper budgets')
print(f'🔀 Split       : 80% Train | 20% Test')
print()
print('📊 Feature Correlations with Sales:')
print(f'   TV        : {tv_corr:.4f}  ← Strongest')
print(f'   Radio     : {rad_corr:.4f}')
print(f'   Newspaper : {news_corr:.4f}  ← Weakest')
print()
print('🤖 Models Trained:')
for name, res in results.items():
    print(f'   • {name:22s} → R²: {res["R² Score"]}')
print()
print(f'🏆 Best Model  : {best_name}')
print(f'🎯 Best R²     : {best_r2}')
print()
print('🔍 Key Findings:')
print('   • TV advertising has strongest impact on sales')
print('   • Radio has moderate positive effect')
print('   • Newspaper has very weak impact on sales')
print('   • TV + Radio combination gives best ROI')
print()
print('👨‍💻 Author : Raj Chakrawarti | CodeAlpha Internship | Task 4')
print('=' * 60)